In [21]:
import akshare as ak
import numpy as np
import pandas as pd
import akshare as ak



In [22]:
useful_codes = pd.DataFrame()
code_list = pd.DataFrame(columns=['代码', '名称'])


In [ ]:
stock_sh_a_spot_em_df = ak.stock_sh_a_spot_em()
stock_sz_a_spot_em_df = ak.stock_sz_a_spot_em()
stock_sh_a_spot_em_df.dropna(inplace=True)
stock_sz_a_spot_em_df.dropna(inplace=True)

code_list = pd.concat([code_list, stock_sh_a_spot_em_df[['代码', '名称']]], ignore_index=True)
code_list = pd.concat([code_list, stock_sz_a_spot_em_df[['代码', '名称']]], ignore_index=True)
code_list

In [ ]:
code_list.to_excel('code_list.xlsx', index=True)

In [39]:
code_list = pd.read_excel('code_list.xlsx', dtype={'代码': str})

In [40]:
def calculate_ewma_trend(prices, lambda_value, initial_trend):
    # 初始化 EWMA 趋势序列

    returns = prices.pct_change().dropna()  # 使用 pandas 的 pct_change 计算收益率
    ewma_trend = pd.Series(np.zeros_like(returns), index=returns.index)
    
    ewma_trend.iloc[0] = initial_trend  # 设置初始趋势值
    
    # 递推计算 EWMA 趋势
    for t in range(1, len(returns)):
       ewma_trend.iloc[t] = (1 - lambda_value) * ewma_trend.iloc[t-1] + lambda_value * returns.iloc[t]

    return ewma_trend

In [41]:
lambda_value = 0.2  # EWMA 参数
EWMA_threshold = 0.2
initial_trend=0

In [44]:
for code in code_list['代码']:
    try:
        source_data = ak.stock_zh_a_hist(symbol=code, period="daily", start_date="20240306", end_date='20250306', adjust='')
        source_data['ewma_trend'] = calculate_ewma_trend(source_data['收盘'], lambda_value, initial_trend)
        useful_codes = source_data.loc[abs(source_data['ewma_trend']) > EWMA_threshold, '股票代码'].tolist()

        if useful_codes:
            print(useful_codes)
    except KeyError as e:
        print(f"股票代码 {code} 出现错误：{e}")
    except Exception as e:
        print(f"股票代码 {code} 出现未知错误：{e}")

KeyboardInterrupt: 

In [38]:
source_data

,日期,股票代码,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率,ewma_trend
0,2024-03-06,688187,43.99,43.81,44.75,43.68,53951,238329018.0,2.43,-0.66,-0.29,2.08,NaN
1,2024-03-07,688187,43.96,44.73,46.68,43.90,147548,672064537.0,6.35,2.10,0.92,5.68,0.000000
2,2024-03-08,688187,44.28,44.76,45.12,43.68,79661,355559842.0,3.22,0.07,0.03,3.06,0.000134
3,2024-03-11,688187,45.00,44.82,46.60,44.33,81251,367584946.0,5.07,0.13,0.06,3.13,0.000375
4,2024-03-12,688187,45.00,43.32,45.00,43.09,89928,392679169.0,4.26,-3.35,-1.50,3.46,-0.006393
...,...,...,...,...,...,...,...,...,...,...,...,...,...
237,2025-02-28,688187,47.40,46.39,47.60,46.10,92731,433736587.0,3.21,-0.81,-0.38,3.32,0.007694
238,2025-03-03,688187,46.20,46.89,48.30,46.20,81992,388793702.0,4.53,1.08,0.50,2.94,0.008311
239,2025-03-04,688187,46.70,46.56,47.47,46.40,58404,273572541.0,2.28,-0.70,-0.33,2.09,0.005241
240,2025-03-05,688187,46.34,46.27,46.62,45.73,48027,221278101.0,1.91,-0.62,-0.29,1.72,0.002947
